In [1]:
# ================================================================
# STEP 7
# FINAL HYBRID AI CAREER RECOMMENDATION ENGINE
#
# Components:
# 1. Technical Skill Matching
# 2. Interest Matching
# 3. Education / Branch Matching
# 4. Aptitude Matching
# 5. Industry Matching
# 6. Experience Matching
# 7. TF-IDF + Cosine Similarity
# ================================================================

import pandas as pd
import numpy as np
import os
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ================================================================
# 1. PROJECT PATHS
# ================================================================

BASE_PATH = r"C:\Users\shiva\Desktop\AI_Career_Recommendation_System"

CAREER_PATH = os.path.join(
    BASE_PATH,
    "data",
    "career_knowledge_base.csv"
)

JOB_PATH = os.path.join(
    BASE_PATH,
    "data",
    "processed_jobs.csv"
)

OUTPUT_PATH = os.path.join(
    BASE_PATH,
    "data",
    "final_recommendation_test.csv"
)


# ================================================================
# 2. LOAD DATA
# ================================================================

careers = pd.read_csv(CAREER_PATH)
jobs = pd.read_csv(JOB_PATH)

print("=" * 70)
print("FINAL HYBRID AI CAREER RECOMMENDATION ENGINE")
print("=" * 70)

print("\nCareer Knowledge Base:", careers.shape)
print("Job Dataset          :", jobs.shape)


# ================================================================
# 3. HELPER FUNCTIONS
# ================================================================

def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def convert_to_set(text):

    if pd.isna(text):
        return set()

    return {
        clean_text(item)
        for item in str(text).split(",")
        if clean_text(item)
    }


def level_score(user_level, required_level):

    levels = {
        "low": 1,
        "medium": 2,
        "high": 3
    }

    user = levels.get(
        clean_text(user_level),
        1
    )

    required = levels.get(
        clean_text(required_level),
        1
    )

    if user >= required:
        return 100

    elif user == required - 1:
        return 60

    else:
        return 30


# ================================================================
# 4. TECHNICAL SKILLS
# ================================================================

technical_skills = {

    "aws",
    "c++",
    "content writing",
    "curriculum design",
    "customer service",
    "edtech",
    "excel",
    "financial modeling",
    "google ads",
    "java",
    "machine learning",
    "market research",
    "medical research",
    "merchandising",
    "nursing",
    "patient care",
    "pharmaceuticals",
    "production planning",
    "python",
    "quality control",
    "react",
    "research",
    "risk analysis",
    "seo",
    "sql",
    "sales",
    "social media",
    "supply chain",
    "teaching"
}


# ================================================================
# 5. BUILD CAREER DOCUMENTS FOR TF-IDF
# ================================================================

career_documents = []

for _, row in careers.iterrows():

    document = " ".join([

        clean_text(row["career"]),
        clean_text(row["category"]),
        clean_text(row["industries"]),
        clean_text(row["skills"]),
        clean_text(row["interests"]),
        clean_text(row["education"]),
        clean_text(row["programming_level"]),
        clean_text(row["mathematics_level"]),
        clean_text(row["communication_level"]),
        clean_text(row["problem_solving_level"]),
        clean_text(row["creativity_level"]),
        clean_text(row["leadership_level"])

    ])

    career_documents.append(document)


# ================================================================
# 6. CREATE TF-IDF MODEL
# ================================================================

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    stop_words="english"
)

career_vectors = vectorizer.fit_transform(
    career_documents
)

print("\nTF-IDF Matrix Shape:")
print(career_vectors.shape)


# ================================================================
# 7. RECOMMENDATION FUNCTION
# ================================================================

def recommend_careers(

    education,
    branch,
    interests,
    skills,

    programming,
    mathematics,
    communication,
    problem_solving,
    creativity,
    leadership,

    preferred_industry,
    experience

):

    # ------------------------------------------------------------
    # CLEAN USER DATA
    # ------------------------------------------------------------

    education = clean_text(education)
    branch = clean_text(branch)
    preferred_industry = clean_text(
        preferred_industry
    )
    experience = clean_text(
        experience
    )

    interests = {
        clean_text(x)
        for x in interests
        if clean_text(x)
    }

    skills = {
        clean_text(x)
        for x in skills
        if clean_text(x)
    }

    # ------------------------------------------------------------
    # CREATE STUDENT TEXT PROFILE
    # ------------------------------------------------------------

    student_text = " ".join([

        education,
        branch,

        " ".join(interests),

        " ".join(skills),

        f"programming {clean_text(programming)}",

        f"mathematics {clean_text(mathematics)}",

        f"communication {clean_text(communication)}",

        f"problem solving {clean_text(problem_solving)}",

        f"creativity {clean_text(creativity)}",

        f"leadership {clean_text(leadership)}",

        preferred_industry,
        experience

    ])

    student_text = clean_text(
        student_text
    )

    # ------------------------------------------------------------
    # TF-IDF STUDENT VECTOR
    # ------------------------------------------------------------

    student_vector = vectorizer.transform(
        [student_text]
    )

    similarity_scores = cosine_similarity(
        student_vector,
        career_vectors
    )[0]

    # ------------------------------------------------------------
    # RESULTS
    # ------------------------------------------------------------

    results = []

    # ============================================================
    # LOOP THROUGH CAREERS
    # ============================================================

    for index, career in careers.iterrows():

        # --------------------------------------------------------
        # CAREER INFORMATION
        # --------------------------------------------------------

        career_skills = convert_to_set(
            career["skills"]
        )

        career_interests = convert_to_set(
            career["interests"]
        )

        career_education = convert_to_set(
            career["education"]
        )

        career_industries = convert_to_set(
            career["industries"]
        )

        # --------------------------------------------------------
        # 1. TECHNICAL SKILL SCORE
        # --------------------------------------------------------

        career_technical_skills = (
            career_skills
            .intersection(
                technical_skills
            )
        )

        user_technical_skills = (
            skills
            .intersection(
                technical_skills
            )
        )

        matching_skills = (
            user_technical_skills
            .intersection(
                career_technical_skills
            )
        )

        missing_skills = (
            career_technical_skills
            -
            user_technical_skills
        )

        if len(career_technical_skills) > 0:

            skill_score = (

                len(matching_skills)
                /
                len(career_technical_skills)

            ) * 100

        else:

            skill_score = 0


        # --------------------------------------------------------
        # 2. INTEREST SCORE
        # --------------------------------------------------------

        matching_interests = (
            interests
            .intersection(
                career_interests
            )
        )

        if len(career_interests) > 0:

            interest_score = (

                len(matching_interests)
                /
                len(career_interests)

            ) * 100

        else:

            interest_score = 0


        # --------------------------------------------------------
        # 3. EDUCATION / BRANCH SCORE
        # --------------------------------------------------------

        education_score = 0

        education_keywords = [
            education,
            branch
        ]

        for item in career_education:

            for keyword in education_keywords:

                if keyword and (
                    keyword in item
                    or item in keyword
                ):

                    education_score = 100
                    break

            if education_score == 100:
                break


        # Common engineering compatibility

        if education_score == 0:

            engineering_branches = [
                "cse",
                "computer",
                "aiml",
                "artificial intelligence",
                "information technology",
                "it"
            ]

            if any(
                b in branch
                for b in engineering_branches
            ):

                education_score = 80


        # --------------------------------------------------------
        # 4. APTITUDE SCORE
        # --------------------------------------------------------

        programming_score = level_score(
            programming,
            career["programming_level"]
        )

        mathematics_score = level_score(
            mathematics,
            career["mathematics_level"]
        )

        communication_score = level_score(
            communication,
            career["communication_level"]
        )

        problem_solving_score = level_score(
            problem_solving,
            career["problem_solving_level"]
        )

        creativity_score = level_score(
            creativity,
            career["creativity_level"]
        )

        leadership_score = level_score(
            leadership,
            career["leadership_level"]
        )

        aptitude_score = np.mean([

            programming_score,
            mathematics_score,
            communication_score,
            problem_solving_score,
            creativity_score,
            leadership_score

        ])


        # --------------------------------------------------------
        # 5. INDUSTRY SCORE
        # --------------------------------------------------------

        if preferred_industry in career_industries:

            industry_score = 100

        else:

            industry_score = 30


        # --------------------------------------------------------
        # 6. EXPERIENCE SCORE
        # --------------------------------------------------------

        if experience in [
            "student",
            "entry level"
        ]:

            experience_score = 100

        elif experience == "mid level":

            experience_score = 70

        elif experience == "senior level":

            experience_score = 60

        else:

            experience_score = 70


        # --------------------------------------------------------
        # 7. NLP SIMILARITY SCORE
        # --------------------------------------------------------

        nlp_score = (
            similarity_scores[index]
            * 100
        )


        # ========================================================
        # 8. FINAL HYBRID SCORE
        # ========================================================

        final_score = (

            skill_score * 0.30

            + interest_score * 0.20

            + education_score * 0.15

            + aptitude_score * 0.15

            + industry_score * 0.10

            + experience_score * 0.05

            + nlp_score * 0.05

        )


        # --------------------------------------------------------
        # 9. SALARY INFORMATION
        # --------------------------------------------------------

        career_name = clean_text(
            career["career"]
        )

        career_words = [

            word
            for word in career_name.split()
            if len(word) > 3

        ]

        if career_words:

            matching_jobs = jobs[
                jobs["job_title"]
                .fillna("")
                .apply(

                    lambda title:
                    sum(

                        word in clean_text(title)

                        for word in career_words

                    ) >= 1

                )
            ]

        else:

            matching_jobs = pd.DataFrame()


        if len(matching_jobs) > 0:

            average_salary = (
                matching_jobs["salary"]
                .mean()
            )

            minimum_salary = (
                matching_jobs["salary"]
                .min()
            )

            maximum_salary = (
                matching_jobs["salary"]
                .max()
            )

        else:

            average_salary = (
                jobs["salary"].mean()
            )

            minimum_salary = (
                jobs["salary"].min()
            )

            maximum_salary = (
                jobs["salary"].max()
            )


        # --------------------------------------------------------
        # 10. SAVE RESULT
        # --------------------------------------------------------

        results.append({

            "career":
                career["career"],

            "category":
                career["category"],

            "final_score":
                round(
                    final_score,
                    2
                ),

            "skill_score":
                round(
                    skill_score,
                    2
                ),

            "interest_score":
                round(
                    interest_score,
                    2
                ),

            "education_score":
                round(
                    education_score,
                    2
                ),

            "aptitude_score":
                round(
                    aptitude_score,
                    2
                ),

            "industry_score":
                round(
                    industry_score,
                    2
                ),

            "nlp_similarity":
                round(
                    nlp_score,
                    2
                ),

            "matching_skills":
                ", ".join(
                    sorted(
                        matching_skills
                    )
                ),

            "skills_to_improve":
                ", ".join(
                    sorted(
                        missing_skills
                    )
                ),

            "average_salary":
                round(
                    average_salary,
                    2
                ),

            "minimum_salary":
                round(
                    minimum_salary,
                    2
                ),

            "maximum_salary":
                round(
                    maximum_salary,
                    2
                )
        })


    # ============================================================
    # 11. SORT RESULTS
    # ============================================================

    result_df = pd.DataFrame(
        results
    )

    result_df = result_df.sort_values(
        by="final_score",
        ascending=False
    ).reset_index(
        drop=True
    )

    return result_df


# ================================================================
# 12. TEST STUDENT
# ================================================================

test_result = recommend_careers(

    education="B.Tech",

    branch="AIML",

    interests=[
        "Artificial Intelligence",
        "Machine Learning",
        "Data Science"
    ],

    skills=[
        "Python",
        "SQL",
        "Machine Learning",
        "Research"
    ],

    programming="High",

    mathematics="High",

    communication="Medium",

    problem_solving="High",

    creativity="Medium",

    leadership="Medium",

    preferred_industry="Software",

    experience="Student"
)


# ================================================================
# 13. DISPLAY TOP 5
# ================================================================

print("\n" + "=" * 70)
print("FINAL TOP 5 CAREER RECOMMENDATIONS")
print("=" * 70)

display(

    test_result[
        [
            "career",
            "category",
            "final_score",
            "skill_score",
            "interest_score",
            "education_score",
            "aptitude_score",
            "industry_score",
            "nlp_similarity"
        ]
    ].head(5)

)


# ================================================================
# 14. DETAILED EXPLANATION
# ================================================================

print("\n" + "=" * 70)
print("CAREER EXPLANATION")
print("=" * 70)

for i in range(
    min(5, len(test_result))
):

    row = test_result.iloc[i]

    print("\n" + "-" * 65)

    print(
        f"Rank {i + 1}: "
        f"{row['career']}"
    )

    print(
        f"Final Match Score: "
        f"{row['final_score']}%"
    )

    print(
        f"Technical Skill Match: "
        f"{row['skill_score']}%"
    )

    print(
        f"Interest Match: "
        f"{row['interest_score']}%"
    )

    print(
        f"Education Match: "
        f"{row['education_score']}%"
    )

    print(
        f"Aptitude Match: "
        f"{row['aptitude_score']}%"
    )

    print(
        f"Industry Match: "
        f"{row['industry_score']}%"
    )

    print(
        f"NLP Similarity: "
        f"{row['nlp_similarity']}%"
    )

    print(
        f"Matching Skills: "
        f"{row['matching_skills']}"
    )

    print(
        f"Skills to Improve: "
        f"{row['skills_to_improve']}"
    )

    print(
        f"Average Salary: "
        f"{row['average_salary']}"
    )


# ================================================================
# 15. SAVE FINAL TEST RESULTS
# ================================================================

test_result.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\n" + "=" * 70)

print(
    "Final results saved to:"
)

print(
    OUTPUT_PATH
)

print("=" * 70)

print(
    "\n✅ STEP 7 COMPLETED SUCCESSFULLY!"
)

FINAL HYBRID AI CAREER RECOMMENDATION ENGINE

Career Knowledge Base: (11, 13)
Job Dataset          : (50000, 13)

TF-IDF Matrix Shape:
(11, 270)

FINAL TOP 5 CAREER RECOMMENDATIONS


,career,category,final_score,skill_score,interest_score,education_score,aptitude_score,industry_score,nlp_similarity
0,AI / Machine Learning Engineer,Artificial Intelligence,82.09,80.00,50.0,100,100.00,100,61.83
1,Data Scientist,Data Science,81.50,80.00,50.0,100,100.00,100,50.03
2,Research Scientist,Research,81.09,100.00,25.0,100,93.33,100,41.72
3,Cybersecurity Analyst,Cybersecurity,76.01,100.00,0.0,100,100.00,100,20.13
4,Software Developer,Software Development,66.12,66.67,0.0,100,100.00,100,22.45



CAREER EXPLANATION

-----------------------------------------------------------------
Rank 1: AI / Machine Learning Engineer
Final Match Score: 82.09%
Technical Skill Match: 80.0%
Interest Match: 50.0%
Education Match: 100%
Aptitude Match: 100.0%
Industry Match: 100%
NLP Similarity: 61.83%
Matching Skills: machine learning, python, research, sql
Skills to Improve: aws
Average Salary: 94971.88

-----------------------------------------------------------------
Rank 2: Data Scientist
Final Match Score: 81.5%
Technical Skill Match: 80.0%
Interest Match: 50.0%
Education Match: 100%
Aptitude Match: 100.0%
Industry Match: 100%
NLP Similarity: 50.03%
Matching Skills: machine learning, python, research, sql
Skills to Improve: excel
Average Salary: 94840.61

-----------------------------------------------------------------
Rank 3: Research Scientist
Final Match Score: 81.09%
Technical Skill Match: 100.0%
Interest Match: 25.0%
Education Match: 100%
Aptitude Match: 93.33%
Industry Match: 100%
NLP

In [2]:
# ================================================================
# STEP 8: MULTI-PROFILE AI RECOMMENDATION TESTING
# ================================================================

print("=" * 80)
print("MULTI-PROFILE RECOMMENDATION TEST")
print("=" * 80)


# ================================================================
# PROFILE 1 — AIML / AI STUDENT
# ================================================================

profile_1 = recommend_careers(

    education="B.Tech",

    branch="AIML",

    interests=[
        "Artificial Intelligence",
        "Machine Learning",
        "Data Science"
    ],

    skills=[
        "Python",
        "SQL",
        "Machine Learning",
        "Research"
    ],

    programming="High",
    mathematics="High",
    communication="Medium",
    problem_solving="High",
    creativity="Medium",
    leadership="Medium",

    preferred_industry="Software",

    experience="Student"
)


# ================================================================
# PROFILE 2 — SOFTWARE / CSE STUDENT
# ================================================================

profile_2 = recommend_careers(

    education="B.Tech",

    branch="CSE",

    interests=[
        "Programming",
        "Software Development",
        "Web Development"
    ],

    skills=[
        "Python",
        "Java",
        "C++",
        "SQL",
        "React"
    ],

    programming="High",
    mathematics="Medium",
    communication="Medium",
    problem_solving="High",
    creativity="High",
    leadership="Low",

    preferred_industry="Software",

    experience="Student"
)


# ================================================================
# PROFILE 3 — BUSINESS / FINANCE STUDENT
# ================================================================

profile_3 = recommend_careers(

    education="B.Sc",

    branch="Commerce",

    interests=[
        "Finance",
        "Business",
        "Investment",
        "Analytics"
    ],

    skills=[
        "Excel",
        "Financial Modeling",
        "Risk Analysis",
        "Market Research",
        "SQL"
    ],

    programming="Low",
    mathematics="High",
    communication="High",
    problem_solving="High",
    creativity="Medium",
    leadership="High",

    preferred_industry="Finance",

    experience="Student"
)


# ================================================================
# PROFILE 4 — CREATIVE / COMMUNICATION STUDENT
# ================================================================

profile_4 = recommend_careers(

    education="B.Sc",

    branch="General",

    interests=[
        "Marketing",
        "Social Media",
        "Content Creation",
        "Business"
    ],

    skills=[
        "SEO",
        "Google Ads",
        "Content Writing",
        "Social Media",
        "Market Research"
    ],

    programming="Low",
    mathematics="Low",
    communication="High",
    problem_solving="Medium",
    creativity="High",
    leadership="High",

    preferred_industry="Marketing",

    experience="Student"
)


# ================================================================
# FUNCTION TO DISPLAY TOP 5
# ================================================================

def show_profile_results(profile, profile_name):

    print("\n")
    print("=" * 80)
    print(profile_name)
    print("=" * 80)

    display(

        profile[
            [
                "career",
                "final_score",
                "skill_score",
                "interest_score",
                "education_score",
                "aptitude_score",
                "industry_score",
                "nlp_similarity"
            ]
        ].head(5)

    )


# ================================================================
# DISPLAY ALL PROFILES
# ================================================================

show_profile_results(
    profile_1,
    "PROFILE 1 — AIML / AI STUDENT"
)

show_profile_results(
    profile_2,
    "PROFILE 2 — SOFTWARE / CSE STUDENT"
)

show_profile_results(
    profile_3,
    "PROFILE 3 — BUSINESS / FINANCE STUDENT"
)

show_profile_results(
    profile_4,
    "PROFILE 4 — CREATIVE / MARKETING STUDENT"
)


# ================================================================
# CHECK TOP RECOMMENDATIONS
# ================================================================

print("\n")
print("=" * 80)
print("TOP RECOMMENDATION FOR EACH PROFILE")
print("=" * 80)


profiles = {

    "AIML / AI Student":
        profile_1,

    "Software / CSE Student":
        profile_2,

    "Business / Finance Student":
        profile_3,

    "Creative / Marketing Student":
        profile_4
}


for name, result in profiles.items():

    top = result.iloc[0]

    print(
        f"\n{name}"
    )

    print(
        f"→ Recommended Career: "
        f"{top['career']}"
    )

    print(
        f"→ Match Score: "
        f"{top['final_score']}%"
    )


print("\n")
print("=" * 80)
print("✅ MULTI-PROFILE TEST COMPLETED!")
print("=" * 80)

MULTI-PROFILE RECOMMENDATION TEST


PROFILE 1 — AIML / AI STUDENT


,career,final_score,skill_score,interest_score,education_score,aptitude_score,industry_score,nlp_similarity
0,AI / Machine Learning Engineer,82.09,80.00,50.0,100,100.00,100,61.83
1,Data Scientist,81.50,80.00,50.0,100,100.00,100,50.03
2,Research Scientist,81.09,100.00,25.0,100,93.33,100,41.72
3,Cybersecurity Analyst,76.01,100.00,0.0,100,100.00,100,20.13
4,Software Developer,66.12,66.67,0.0,100,100.00,100,22.45




PROFILE 2 — SOFTWARE / CSE STUDENT


,career,final_score,skill_score,interest_score,education_score,aptitude_score,industry_score,nlp_similarity
0,Software Developer,91.23,100.00,66.67,100,100.00,100,57.98
1,Web Developer,87.20,100.00,50.00,100,100.00,100,43.93
2,Cloud / DevOps Engineer,69.63,66.67,25.00,100,93.33,100,12.64
3,Cybersecurity Analyst,64.98,66.67,0.00,100,93.33,100,19.56
4,Research Scientist,58.79,50.00,0.00,100,86.67,100,15.73




PROFILE 3 — BUSINESS / FINANCE STUDENT


,career,final_score,skill_score,interest_score,education_score,aptitude_score,industry_score,nlp_similarity
0,Financial Analyst,97.94,100.0,100.0,100,100.00,100,58.79
1,Business Analyst,85.97,100.0,50.0,100,93.33,100,39.31
2,Data Analyst,74.41,80.0,25.0,100,93.33,100,28.29
3,Data Scientist,56.00,40.0,0.0,100,88.33,100,14.93
4,Research Scientist,50.59,25.0,0.0,100,81.67,100,16.88




PROFILE 4 — CREATIVE / MARKETING STUDENT


,career,final_score,skill_score,interest_score,education_score,aptitude_score,industry_score,nlp_similarity
0,Digital Marketing Specialist,98.26,100.0,100.0,100,100.00,100,65.23
1,Business Analyst,55.32,25.0,25.0,100,80.00,100,16.36
2,Financial Analyst,53.90,20.0,25.0,100,81.67,100,12.94
3,Data Analyst,52.90,20.0,25.0,100,75.00,100,12.95
4,Web Developer,41.68,0.0,0.0,100,75.00,100,8.56




TOP RECOMMENDATION FOR EACH PROFILE

AIML / AI Student
→ Recommended Career: AI / Machine Learning Engineer
→ Match Score: 82.09%

Software / CSE Student
→ Recommended Career: Software Developer
→ Match Score: 91.23%

Business / Finance Student
→ Recommended Career: Financial Analyst
→ Match Score: 97.94%

Creative / Marketing Student
→ Recommended Career: Digital Marketing Specialist
→ Match Score: 98.26%


✅ MULTI-PROFILE TEST COMPLETED!
